In [1]:
//| code-fold: true

%mavenRepo scijava.public https://maven.scijava.org/content/groups/public
%maven org.scijava:scijava-common:2.97.0
%maven net.imglib2:imglib2:7.1.2
%maven net.imglib2:imglib2-algorithm:0.16.0
%maven net.imglib2:imglib2-cache:1.0.0-beta-18

import net.imglib2.*;
import net.imglib2.converter.*;
import net.imglib2.img.array.*;
import net.imglib2.img.cell.*;
import net.imglib2.type.numeric.real.*;
import net.imglib2.type.numeric.integer.*;
import net.imglib2.view.*;
import net.imglib2.util.*;

import net.imglib2.blocks.*;
import net.imglib2.algorithm.blocks.*;
import net.imglib2.algorithm.blocks.downsample.*;
import net.imglib2.algorithm.gauss3.*;

import net.imglib2.cache.img.*;

public static long timeIt( Runnable r ) {
    long start = System.currentTimeMillis();
    r.run();
    return System.currentTimeMillis() - start;
}

This post summarizes content from the following pull requests:

* [PrimitiveBlocks copies RandomAccessible blocks to primitive arrays](https://github.com/imglib/imglib2/pull/330)
* [Block algorithms: type conversion and downsampling](https://github.com/imglib/imglib2-algorithm/pull/97)
* [BlockSupplier API and block algorithm improvements](https://github.com/imglib/imglib2-algorithm/pull/104)

## 1 PrimitiveBlocks

Imglib2's `PrimitiveBlocks` adds functionality to extract blocks from RandomAccessible into flat primitive arrays.

This will be useful for example to copy data to [Tensor](https://github.com/bioimage-io/model-runner-java/blob/main/src/main/java/io/bioimage/modelrunner/tensor/Tensor.java), to extract blocks for storing into [N5](https://github.com/saalfeldlab/n5), for interfacing CLIJ, for running small algorithm kernels directly on primitive arrays, etc...

In [3]:
// Some Img
CellImg< UnsignedByteType, ? > cellImg3D = new CellImgFactory(
    new UnsignedByteType()).create(60, 60, 60);

// A potentially complicated view of an Img
RandomAccessible< FloatType > view = Converters.convert(
    Views.extendBorder(
        Views.hyperSlice(
            Views.zeroMin(
                Views.rotate( cellImg3D, 1, 0 )
            ),
            2, 80 )
    ),
    new RealFloatConverter<>(),
    new FloatType()
);

// Copy the data into a PrimitiveBlock (float[]) in order to optimize processing

// allocate destination array
float[] data = new float[ 40 * 50 ];

// starting position and size for copy operation
int[] position = new int[]{ 10, 20 };
int[] size = new int[]{ 40, 50 };

// copy from 'view' into 'data'
PrimitiveBlocks< FloatType > blocks = PrimitiveBlocks.of( view );

long timeBlockCopy = timeIt( () -> {
    blocks.copy( position, data, size );
});
System.out.println("Block " + timeBlockCopy + "ms");

6


The idea of the optimized copier is:

Instead of using `RandomAccess` that checks for every pixel whether it enters a new `Cell`, whether it is out-of-bounds, etc., all these checks are precomputed and then relevant data from each `Cell` is copied in one go. The speedup can be dramatic, in particular if the underlying source data is in a `CellImg`.

Some benchmarks included, here is for example results of [`CopyBenchmarkViewPrimitiveBlocks`](https://github.com/imglib/imglib2/blob/1fd92e5385127f25d27695771f001810c6e11109/src/test/java/net/imglib2/blocks/CopyBenchmarkViewPrimitiveBlocks.java):

```
# JMH version: 1.35
# VM version: JDK 17.0.3, OpenJDK 64-Bit Server VM, 17.0.3+7-LTS
...

Benchmark                                                  (oob)  (permute)  Mode  Cnt   Score   Error  Units
CopyBenchmarkViewPrimitiveBlocks.benchmarkLoopBuilder       true       true  avgt    5  12,789 ± 0,285  ms/op
CopyBenchmarkViewPrimitiveBlocks.benchmarkLoopBuilder       true      false  avgt    5   9,682 ± 0,152  ms/op
CopyBenchmarkViewPrimitiveBlocks.benchmarkLoopBuilder      false       true  avgt    5  14,333 ± 0,099  ms/op
CopyBenchmarkViewPrimitiveBlocks.benchmarkLoopBuilder      false      false  avgt    5  12,721 ± 0,123  ms/op
CopyBenchmarkViewPrimitiveBlocks.benchmarkPrimitiveBlocks   true       true  avgt    5   0,541 ± 0,010  ms/op
CopyBenchmarkViewPrimitiveBlocks.benchmarkPrimitiveBlocks   true      false  avgt    5   0,315 ± 0,024  ms/op
CopyBenchmarkViewPrimitiveBlocks.benchmarkPrimitiveBlocks  false       true  avgt    5   0,570 ± 0,013  ms/op
CopyBenchmarkViewPrimitiveBlocks.benchmarkPrimitiveBlocks  false      false  avgt    5   0,322 ± 0,008  ms/op
```



If a source `RandomAccessible` cannot be understood, `PrimitiveBlocks.of(...)` will return
a fall-back implementation (based on `LoopBuilder`). With the optional `OnFallback` argument
of `PrimitiveBlocks.of(...)` it can be configured whether fall-back should be

* silently accepted (`ACCEPT`),
* a warning should be printed (`WARN`) -- the default,
* or an IllegalArgumentException thrown (`FAIL`).
    * The warning/exception message explains why the source `RandomAccessible` requires fall-back.

The only really un-supported case is if the pixel type `T` does not map one-to-one to a 
primitive type. For example, `ComplexDoubleType` or `Unsigned4BitType` are not supported 
(at least not yet).

`PrimitiveBlocks.copy` is single-threaded, the idea being to parallelize over blocks instead
of  the copying within a block.  `PrimitiveBlocks` is not thread-safe in general, but has
a method `threadSafe()` to obtain a thread-safe instance (implemented using `ThreadLocal` copies). 
For example,

In [3]:
PrimitiveBlocks< FloatType > blocks = PrimitiveBlocks.of( view ).threadSafe();

## 2 Block algorithms

1. A framework for implementing block-processing algorithms that build off the `PrimitiveBlocks` utility introduced in imglib2 6.2.0. Package `net.imglib2.algorithm.block` contains helpers to define and chain block operators. In particular,  the `BlockProcessor<I,O>` interface specifies an algorithm that computes values in a flattened primitive output array (type `O`) from values in a flattened primitive input array (type `I`). To avoid mistakes in unchecked casting of primitive array types, `BlockProcessor` should be typically wrapped in `UnaryBlockOperator<S,T>` where `S` and `T` are source and target ImgLib2 `RealType` corresponding to the respective `I` and `O` array types. `UnaryBlockOperator` can be chained using `UnaryBlockOperator.andThen(...)`.
2. An operator to *convert* between common `RealTypes` (signed and unsigned, with and without clamping). This code is partially auto-generated using the `bin/generate.groovy` script lifted from scijava-ops.
3. An operator to *downsample*, by factor 2, blocks of common `RealType`. It can be specified whether intermediate calculations should happen in `FLOAT` or `DOUBLE` precision (or `AUTO`matically determined). There are versions for downsampling 2x2x2... blocks with half-pixel offset and 3x3x3... blocks on pixel centers.

Together with `PrimitiveBlocks` from imglib2 core, the downsampling algorithm achieves speedup of factor ~4 over the downsampling used in bigdataviewer-core, and speedup of factor ~10 to ~30 over the downsampling used in BigStitcher (depending on Java version, block size, etc). (This is on `ArrayImg` inputs. Speedups are probably even larger for `CellImg` or `PlanarImg` inputs, where `PrimitveBlocks` has a larger impact.)

In [4]:
public static RandomAccessibleInterval<FloatType> downsampleGauss(RandomAccessibleInterval<FloatType> img) { 
    
    final boolean[] downsampleInDim = { true, true, true };
    final long[] downsampledDimensions = Downsample.getDownsampledDimensions( 
        img.dimensionsAsLongArray(), downsampleInDim );
        
    final CellImg<FloatType, ?> tgt = new CellImgFactory<FloatType>(new FloatType(), cellDimensions)
        .create(downsampledDimensions);

    Gauss3.gauss(1.0, Views.extendBorder(img), tgt);
    return tgt;
}

public static RandomAccessibleInterval<FloatType> downsampleBlocks(RandomAccessibleInterval<FloatType> img) { 
    
    final boolean[] downsampleInDim = { true, true, true };
    final long[] downsampledDimensions = Downsample.getDownsampledDimensions( 
        img.dimensionsAsLongArray(), downsampleInDim );
        
    final BlockSupplier< FloatType > blocks = BlockSupplier.of( Views.extendBorder( img ) );
    final CachedCellImg< FloatType, ? > downsampled = BlockAlgoUtils.cellImg(
                    blocks.andThen( Downsample.downsample(
                            ComputationType.DOUBLE,
                            Downsample.Offset.CENTERED )
                    ),
                    downsampledDimensions, cellDimensions );

    return downsampled;
}

final int[] cellDimensions = { 64, 64, 64 };
final CellImg<FloatType, ?> img = new CellImgFactory<FloatType>(new FloatType(), cellDimensions)
                .create(256, 256, 256);
img.forEach(x -> x.setReal(Math.random()));

long timeGauss = timeIt( () -> {
    downsampleGauss( img );
});
System.out.println("Gaussian downsampling took " + timeGauss + "ms");

long timeBlocks = timeIt( () -> {
    downsampleBlocks( img );
});
System.out.println("Blocks downsampling " + timeBlocks + "ms");

Gaussian downsampling took 216ms
Blocks downsampling 12ms
